In [1]:
import pandas as pd
import numpy as np


In [3]:
df = pd.read_csv('/content/student_data.csv')
df.head()

,StudentID,Day,Mood,ShirtColor
0,1,1,H,R
1,1,2,H,R
2,1,3,S,B
3,1,4,S,B
4,1,5,H,R


In [4]:
first_day = df[df['Day'] == 1]
P_init = first_day['Mood'].value_counts(normalize=True)
P_init = P_init.to_dict()

print("Initial Probability Distribution (P(M₁)):")
print(P_init)


Initial Probability Distribution (P(M₁)):
{'H': 0.6, 'S': 0.4}


In [5]:
moods = ['H', 'S']
transition_counts = pd.DataFrame(0, index=moods, columns=moods)

for student in df['StudentID'].unique():
    student_data = df[df['StudentID'] == student].sort_values('Day')
    for i in range(len(student_data) - 1):
        m1 = student_data.iloc[i]['Mood']
        m2 = student_data.iloc[i+1]['Mood']
        transition_counts.loc[m1, m2] += 1

transition_probs = transition_counts.div(transition_counts.sum(axis=1), axis=0)
print("Transition Probability Matrix (P(M₂|M₁)):")
print(transition_probs)


Transition Probability Matrix (P(M₂|M₁)):
          H         S
H  0.654545  0.345455
S  0.450000  0.550000


In [6]:
colors = ['R', 'G', 'B']
emission_counts = pd.DataFrame(0, index=moods, columns=colors)

for mood in moods:
    subset = df[df['Mood'] == mood]
    for color in colors:
        emission_counts.loc[mood, color] = len(subset[subset['ShirtColor'] == color])

emission_probs = emission_counts.div(emission_counts.sum(axis=1), axis=0)
print("Emission Probability Matrix (P(C|M)):")
print(emission_probs)


Emission Probability Matrix (P(C|M)):
          R         G         B
H  0.719298  0.280702  0.000000
S  0.000000  0.139535  0.860465


In [7]:
observations = ['R', 'B', 'G']

mood_sequences = [
    ['H','H','H'], ['H','H','S'], ['H','S','H'], ['H','S','S'],
    ['S','H','H'], ['S','H','S'], ['S','S','H'], ['S','S','S']
]

results = {}

for seq in mood_sequences:
    M1, M2, M3 = seq
    P = (
        P_init[M1] *
        emission_probs.loc[M1, observations[0]] *
        transition_probs.loc[M1, M2] *
        emission_probs.loc[M2, observations[1]] *
        transition_probs.loc[M2, M3] *
        emission_probs.loc[M3, observations[2]]
    )
    results[tuple(seq)] = P

print("\nProbabilities for all possible mood sequences:")
for k,v in results.items():
    print(f"{k}: {v:.8f}")

best_seq = max(results, key=results.get)
print("\nMost likely mood sequence:", best_seq)
print("With probability:", results[best_seq])



Probabilities for all possible mood sequences:
('H', 'H', 'H'): 0.00000000
('H', 'H', 'S'): 0.00000000
('H', 'S', 'H'): 0.01620474
('H', 'S', 'S'): 0.00984532
('S', 'H', 'H'): 0.00000000
('S', 'H', 'S'): 0.00000000
('S', 'S', 'H'): 0.00000000
('S', 'S', 'S'): 0.00000000

Most likely mood sequence: ('H', 'S', 'H')
With probability: 0.01620474018026038
